# Practical RAG: bot ka jawab galat? Pehle chunks dekho
**AI Explained · Episode 14 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai.

Is notebook mein koi asli AI nahi chalta, koi API key nahi chahiye. Ek chhota **placement FAQ** hai (36 words), usse **chunks** katenge, aur dekhenge ki sentence beech mein kat jaye to RAG ka jawab kyun galat hota hai. Fix: **overlap**. FAQ aur score ka tareeka teaching ke liye bana hai, asli RAG ka nahi.

In [1]:
DOC = ("College FAQ notes "
       "Resume ki last date 15 December "
       "Mock interview ka din Friday hai "
       "Join karne ki limit 60 din "
       "Placement cell roz 10 baje khulta "
       "Sabhi ke liye. "
       "Resume mein sirf ek page rakho")
W = DOC.split()
print(len(W), "words")

36 words


In [2]:
def chunk(words, size, step):
    return [" ".join(words[i:i + size])
            for i in range(0, len(words), step)]

A = chunk(W, 6, 6)   # 6 words, bina overlap
print(len(A), "chunks")
for c in A:
    print("|", c)

6 chunks
| College FAQ notes Resume ki last
| date 15 December Mock interview ka
| din Friday hai Join karne ki
| limit 60 din Placement cell roz
| 10 baje khulta Sabhi ke liye.
| Resume mein sirf ek page rakho


In [3]:
def score(q, c):      # kitne words same
    a = set(q.lower().split())
    return len(a & set(c.lower().split()))

def top(q, chunks, n=2):
    r = sorted(chunks, reverse=True,
               key=lambda c: score(q, c))
    return r[:n]

q = "Resume ki last date kab hai"
for c in top(q, A):
    print(score(q, c), "|", c)

3 | College FAQ notes Resume ki last
2 | din Friday hai Join karne ki


In [4]:
QA = {
    "Resume ki last date kab hai": "15 December",
    "Mock interview ka din kab hai": "Friday",
    "Join karne ki limit kitne din": "60 din",
    "Placement cell kab khulta hai": "10 baje",
    "Resume mein kitne page chahiye": "ek page",
}

def hits(chunks):    # top chunk mein jawab?
    n = 0
    for q, key in QA.items():
        if key in top(q, chunks, 1)[0]:
            n += 1
    return n

print("bina overlap:", hits(A), "/", len(QA))

bina overlap: 1 / 5


In [5]:
B = chunk(W, 6, 3)   # 3 words overlap
print(len(B), "chunks")
print("overlap 3   :", hits(B), "/", len(QA))

q = "Resume ki last date kab hai"
for c in top(q, B, 1):
    print(score(q, c), "|", c)

12 chunks
overlap 3   : 5 / 5
4 | Resume ki last date 15 December


In [6]:
# OpenAI ke retrieval docs ke default
DEFAULT = {"max_chunk_size_tokens": 800,
           "chunk_overlap_tokens": 400}
for k, v in DEFAULT.items():
    print(f"{k:22} {v}")

max_chunk_size_tokens  800
chunk_overlap_tokens   400


### Asli tools mein
OpenAI ke retrieval docs: "By default, max_chunk_size_tokens is set to 800 and chunk_overlap_tokens is set to 400." Yani overlap koi extra cheez nahi, default hai. Size aur overlap dono settings hain: apne docs par test karke chuno.

OpenAI file search: "the file search call will not return search results by default." Retrieved chunks dekhne hain to unhe include karna padta hai.

Anthropic (Contextual Retrieval): akela chunk context kho sakta hai. Unka example: ek chunk jisme revenue badhna likha hai, par kaun si company aur kab, ye chunk mein nahi. Fix: har chunk ke saath chhota context jodo.

In [7]:
q = "CEO kaun hai"       # doc mein nahi
s = score(q, top(q, B, 1)[0])
print("best score:", s)
if s < 2:             # limit 2, apni chuno
    print("Doc mein nahi hai")
else:
    print(top(q, B, 1)[0])

best score: 1
Doc mein nahi hai


In [8]:
# Apna sawal likho, phir chalao
q = "Mock interview kis din hota hai"
for c in top(q, B, 1):
    print(score(q, c), "|", c)

4 | Mock interview ka din Friday hai
